In [0]:
ruta_resenas = "/Volumes/electrocasa_dev/bronze/landing/resenas/resenas_clientes.json"

resenas = (
    spark.read
        .option("multiLine", "true")
        .json(ruta_resenas)
)

resenas.printSchema()
display(resenas.limit(10))

In [0]:
resenas.createOrReplaceTempView("resenas_tmp")

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT resena_id) AS resenas_unicas,

    SUM(CASE
        WHEN calificacion IS NULL
          OR calificacion < 1
          OR calificacion > 5
        THEN 1 ELSE 0
    END) AS calificacion_invalida,

    SUM(CASE
        WHEN comentario IS NULL
          OR TRIM(comentario) = ''
        THEN 1 ELSE 0
    END) AS comentario_faltante,

    SUM(CASE
        WHEN tags IS NULL
        THEN 1 ELSE 0
    END) AS tags_nulos

FROM resenas_tmp;

In [0]:
%sql

SELECT
    resena_id,
    COUNT(*) AS registros,
    COUNT(DISTINCT concat_ws(
        '||',
        producto_id,
        cliente_id,
        CAST(calificacion AS STRING),
        comentario,
        CAST(tags AS STRING),
        CAST(respuestas AS STRING),
        fecha_resena
    )) AS versiones_distintas
FROM resenas_tmp
GROUP BY resena_id
HAVING COUNT(*) > 1
ORDER BY resena_id;

In [0]:
%sql

SELECT
    COUNT(*) AS total_respuestas,
    SUM(CASE
        WHEN respuesta.autor IS NULL
          OR TRIM(respuesta.autor) = ''
        THEN 1 ELSE 0
    END) AS respuestas_sin_autor
FROM resenas_tmp
LATERAL VIEW explode(respuestas) t AS respuesta;

In [0]:
%sql

SELECT
    COUNT(*) AS filas_huerfanas,
    COUNT(DISTINCT r.resena_id) AS resenas_huerfanas,
    COUNT(DISTINCT r.producto_id) AS productos_huerfanos
FROM resenas_tmp r
LEFT ANTI JOIN (
    SELECT DISTINCT producto_id
    FROM electrocasa_dev.silver.productos
) p
ON r.producto_id = p.producto_id;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT resena_id) AS resenas_unicas,
    COUNT(DISTINCT archivo_origen) AS archivos_origen,
    SUM(CASE
        WHEN _rescued_data IS NOT NULL
        THEN 1 ELSE 0
    END) AS registros_rescatados
FROM electrocasa_dev.bronze.resenas;

In [0]:
%sql

SELECT
    calificacion,
    COUNT(*) AS cantidad
FROM electrocasa_dev.bronze.resenas
WHERE calificacion IS NULL
   OR calificacion < 1
   OR calificacion > 5
GROUP BY calificacion
ORDER BY calificacion;

In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM electrocasa_dev.silver.resenas) AS resenas_validas,

    (SELECT COUNT(*)
     FROM electrocasa_dev.silver.resenas_cuarentena) AS resenas_cuarentena,

    (
        SELECT COUNT(*)
        FROM (
            SELECT producto_huerfano
            FROM electrocasa_dev.silver.resenas

            UNION ALL

            SELECT producto_huerfano
            FROM electrocasa_dev.silver.resenas_cuarentena
        )
        WHERE producto_huerfano = true
    ) AS resenas_huerfanas,

    (
        SELECT COUNT(*)
        FROM (
            SELECT tags
            FROM electrocasa_dev.silver.resenas

            UNION ALL

            SELECT tags
            FROM electrocasa_dev.silver.resenas_cuarentena
        )
        WHERE tags IS NULL
    ) AS tags_nulos;

In [0]:
%sql

WITH todas_resenas AS (
    SELECT respuestas
    FROM electrocasa_dev.silver.resenas

    UNION ALL

    SELECT respuestas
    FROM electrocasa_dev.silver.resenas_cuarentena
)

SELECT
    COUNT(*) AS total_respuestas,
    SUM(CASE
        WHEN respuesta.autor = 'sin_autor'
        THEN 1 ELSE 0
    END) AS respuestas_sin_autor_marcadas,

    SUM(CASE
        WHEN respuesta.autor IS NULL
          OR TRIM(respuesta.autor) = ''
        THEN 1 ELSE 0
    END) AS respuestas_autor_nulo

FROM todas_resenas
LATERAL VIEW explode(respuestas) t AS respuesta;